In [1]:
#import modules and load the data
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from numpy import mean
from numpy import absolute
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
import optuna

df = pd.read_csv("insurance.csv")
df.head()

c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [2]:
#Prepare the data
df = df.dropna() #remove missings
y = df['charges'] #select Target
X = df.drop('charges',axis=1) #Select features
X = pd.get_dummies(X, drop_first=True) #Turn Fuel_Type into dummy variables
columns = X.columns #create index with column names (needed for last step)
scaler = MinMaxScaler() #initiate the scaler
X = scaler.fit_transform(X) #scale the data
X = pd.DataFrame(X,columns=columns) #turn back into a dataframe

In [3]:
#Neural network with cross validation (first attempt)

#First define a function that contains the neural network model
def create_baseline():
    # create model
    nnmodel = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer 
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(1)  # Output layer
    ])
    # Compile model
    nnmodel.compile(loss='mean_absolute_error', #loss is calculated with MAE
                    optimizer='adam', #type of gradient descent
                    metrics=['mae']) #use MAE to evaluate the model
    return nnmodel

# evaluate model 

#define the estimator
estimator = KerasRegressor(build_fn=create_baseline, #you have to use the kerasRegressor wrapper for nn models
                           epochs=200, batch_size=64, #100 epochs and 64 batches
                           verbose=0) #this will shorten the output text
#Define the kfold function
kfold = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42) #I'm using just 5 splits and 1 repeat to save time, but more splits and repeats will be more accurate
#Generate the results
results = cross_val_score(estimator, X, y, cv=kfold,scoring='neg_mean_absolute_error')
#Print the results
print('the average prediction error is: %.0f' % mean(absolute(results)))

c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


the average prediction error is: 1624


In [4]:
#Hyperparameter Tuning (warning, takes at least 15 minutes)

# Define the function to create the model
def build_model(n_layers=5, n_units=64, activation='softplus'): #start with some inital parameters
    nnmodel = tf.keras.models.Sequential() #define the model
    nnmodel.add(tf.keras.layers.Input(shape=(X.shape[1],)))  # Input layer
    for _ in range(n_layers): #create a for loop that goes through the number of layers, number of units, and activation types
        nnmodel.add(tf.keras.layers.Dense(n_units, activation=activation))
    nnmodel.add(tf.keras.layers.Dense(1))  # Output layer
    nnmodel.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mae']) #compile the model and use mean absolute error as measure of fit
    return nnmodel

def objective(trial):
    # Suggest hyperparameters
    n_layers = trial.suggest_categorical('n_layers', [1, 2, 3, 4, 5])         # Number of hidden layers
    n_units = trial.suggest_categorical('n_units', [32, 64, 128, 256, 512])   # Number of nodes per hidden layer
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'softplus', 'elu'])  # activation function
    epochs = trial.suggest_categorical('epochs', [50, 100, 150, 200])         # number of epochs
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])        # number of batches

    # Manual 5-fold CV (Keras models aren't natively supported by cross_val_score)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        nnmodel = build_model(n_layers=n_layers, n_units=n_units, activation=activation)
        nnmodel.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

        val_loss, val_mae = nnmodel.evaluate(X_val, y_val, verbose=0)
        mae_scores.append(val_mae)

    return np.mean(mae_scores)

# search across 10 different combinations
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10)

print("Best parameters found: ", study.best_params) #print the best parameters

[I 2026-10-09 13:19:23,480] A new study created in memory with name: no-name-a134140a-e76d-439a-83a1-7aeff27817d1
[I 2026-10-09 13:20:15,266] Trial 0 finished with value: 12808.218359375 and parameters: {'n_layers': 2, 'n_units': 128, 'activation': 'tanh', 'epochs': 200, 'batch_size': 64}. Best is trial 0 with value: 12808.218359375.
[I 2026-10-09 13:20:55,762] Trial 1 finished with value: 2358.53642578125 and parameters: {'n_layers': 5, 'n_units': 512, 'activation': 'elu', 'epochs': 50, 'batch_size': 64}. Best is trial 1 with value: 2358.53642578125.
[I 2026-10-09 13:21:51,963] Trial 2 finished with value: 3127.147998046875 and parameters: {'n_layers': 2, 'n_units': 256, 'activation': 'relu', 'epochs': 200, 'batch_size': 64}. Best is trial 1 with value: 2358.53642578125.
[I 2026-10-09 13:23:40,938] Trial 3 finished with value: 1652.2571044921874 and parameters: {'n_layers': 5, 'n_units': 64, 'activation': 'relu', 'epochs': 200, 'batch_size': 16}. Best is trial 3 with value: 1652.25710

Best parameters found:  {'n_layers': 5, 'n_units': 256, 'activation': 'relu', 'epochs': 200, 'batch_size': 64}
